In [1]:
from pathlib import Path
import pandas as pd

# Find the data folder beside this notebook, or in the current/parent folder.
candidates = [Path.cwd() / 'data', Path.cwd() / 'outputs' / 'data']
candidates += [parent / 'data' for parent in Path.cwd().parents]
DATA_DIR = next((folder for folder in candidates if (folder / 'workers.csv').exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError('Could not find data/workers.csv. Keep the data folder beside the notebook, or open the extracted Part I project folder in Jupyter.')

workers = pd.read_csv(DATA_DIR / 'workers.csv')
incidents = pd.read_csv(DATA_DIR / 'incidents.csv', parse_dates=['date'])
observations = pd.read_csv(DATA_DIR / 'safety_observations.csv', parse_dates=['date'])
equipment = pd.read_csv(DATA_DIR / 'equipment.csv', parse_dates=['reading_date'])

print('Using sample data from:', DATA_DIR.resolve())
print('Workers:', len(workers))
print('Incidents:', len(incidents))
print('Observations:', len(observations))
print('Equipment:', len(equipment))


Using sample data from: C:\Users\MANDISIKHAYA HLOPHE\OneDrive - University of Witwatersrand\Desktop\data
Workers: 12
Incidents: 6
Observations: 12
Equipment: 10


In [2]:
display(workers.head())
display(incidents.head())
display(observations.head())
display(equipment.head())


,worker_id,department,job_role,shift,ppe_compliant,fatigue_level
0,W001,Underground Operations,Drill Operator,Day,Yes,Low
1,W002,Underground Operations,Loader Operator,Night,No,Medium
2,W003,Processing Plant,Control Room Operator,Day,Yes,Low
3,W004,Maintenance,Fitter,Night,Yes,High
4,W005,Underground Operations,Shift Supervisor,Day,Yes,Medium


,incident_id,date,department,shift,incident_type,severity,injury,status
0,INC001,2026-10-09,Underground Operations,Night,Rockfall,High,No,Open
1,INC002,2026-10-06,Processing Plant,Day,Equipment contact,Medium,Yes,Investigating
2,INC003,2026-10-02,Maintenance,Night,Slip or trip,Low,No,Closed
3,INC004,2026-09-26,Underground Operations,Day,Vehicle interaction,High,Yes,Investigating
4,INC005,2026-09-20,Processing Plant,Night,Chemical exposure,Medium,No,Closed


,observation_id,date,department,worker_id,observation_type,risk_level,status
0,OBS001,2026-10-10,Underground Operations,W002,PPE non-compliance,High,Open
1,OBS002,2026-10-10,Maintenance,W004,Unsafe access near conveyor,Critical,Open
2,OBS003,2026-10-09,Processing Plant,W006,Fatigue reported,High,Open
3,OBS004,2026-10-08,Underground Operations,W005,Loose ground support,Medium,Closed
4,OBS005,2026-10-07,Underground Operations,W008,Near miss: vehicle interaction,Medium,Closed


,equipment_id,equipment_type,department,temperature_c,vibration_mm_s,operating_hours,downtime_hours,scheduled_hours,status,maintenance_due,reading_date
0,TRK-012,Haul truck,Underground Operations,78,9.2,620,30,720,Available,No,2026-10-10
1,LD-004,Loader,Underground Operations,84,4.2,580,54,720,Available,No,2026-10-10
2,DR-021,Drill rig,Underground Operations,102,7.4,410,96,720,Under maintenance,Yes,2026-10-09
3,CV-008,Conveyor,Processing Plant,76,3.1,690,18,720,Available,No,2026-10-10
4,CR-003,Crusher,Processing Plant,96,8.4,510,84,720,Available,Yes,2026-10-10


In [3]:
start_date = min(incidents['date'].min(), observations['date'].min(), equipment['reading_date'].min()).date()
end_date = max(incidents['date'].max(), observations['date'].max(), equipment['reading_date'].max()).date()

incidents_window = incidents[incidents['date'].dt.date.between(start_date, end_date)]
observations_window = observations[observations['date'].dt.date.between(start_date, end_date)]
equipment_window = equipment[equipment['reading_date'].dt.date.between(start_date, end_date)].copy()
near_misses = observations_window['observation_type'].str.contains('near miss', case=False, na=False)
high_risk = observations_window['risk_level'].str.lower().isin(['high', 'critical'])
ppe_pct = workers['ppe_compliant'].str.lower().eq('yes').mean() * 100
equipment_window['availability_pct'] = (equipment_window['operating_hours'] / (equipment_window['operating_hours'] + equipment_window['downtime_hours'])) * 100

kpis = pd.Series({
    'Total workers': len(workers),
    'Total safety incidents': len(incidents_window),
    'Near misses': int(near_misses.sum()),
    'High-risk observations': int(high_risk.sum()),
    'PPE compliance (%)': round(ppe_pct, 1),
    'Total equipment': len(equipment_window),
    'Equipment available': int(equipment_window['status'].str.lower().eq('available').sum()),
    'Equipment under maintenance': int(equipment_window['status'].str.lower().eq('under maintenance').sum()),
    'Average equipment availability (%)': round(equipment_window['availability_pct'].mean(), 1),
})
display(kpis.to_frame('Value'))


,Value
Total workers,12.0
Total safety incidents,6.0
Near misses,2.0
High-risk observations,6.0
PPE compliance (%),75.0
Total equipment,10.0
Equipment available,7.0
Equipment under maintenance,2.0
Average equipment availability (%),88.2


In [4]:
def classify_reading(temperature, vibration):
    alerts = []
    if temperature > 100:
        alerts.append(('CRITICAL', 'Temperature', temperature, '°C'))
    elif temperature >= 80:
        alerts.append(('WARNING', 'Temperature', temperature, '°C'))
    if vibration > 8:
        alerts.append(('CRITICAL', 'Vibration', vibration, 'mm/s'))
    elif vibration >= 5:
        alerts.append(('WARNING', 'Vibration', vibration, 'mm/s'))
    return alerts or [('NORMAL', 'Condition', None, '')]

equipment_window['condition'] = equipment_window.apply(lambda r: classify_reading(r['temperature_c'], r['vibration_mm_s']), axis=1)
equipment_window[['equipment_id', 'temperature_c', 'vibration_mm_s', 'operating_hours', 'downtime_hours', 'availability_pct', 'condition']]


,equipment_id,temperature_c,vibration_mm_s,operating_hours,downtime_hours,availability_pct,condition
0,TRK-012,78,9.2,620,30,95.384615,"[(CRITICAL, Vibration, 9.2, mm/s)]"
1,LD-004,84,4.2,580,54,91.482650,"[(WARNING, Temperature, 84, °C)]"
2,DR-021,102,7.4,410,96,81.027668,"[(CRITICAL, Temperature, 102, °C), (WARNING, V..."
3,CV-008,76,3.1,690,18,97.457627,"[(NORMAL, Condition, None, )]"
4,CR-003,96,8.4,510,84,85.858586,"[(WARNING, Temperature, 96, °C), (CRITICAL, Vi..."
5,BD-015,81,5.6,455,68,86.998088,"[(WARNING, Temperature, 81, °C), (WARNING, Vib..."
6,EX-006,73,4.7,610,22,96.518987,"[(NORMAL, Condition, None, )]"
7,JW-002,88,5.1,390,144,73.033708,"[(WARNING, Temperature, 88, °C), (WARNING, Vib..."
8,CN-011,69,2.8,710,10,98.611111,"[(NORMAL, Condition, None, )]"
9,HT-019,104,8.8,340,110,75.555556,"[(CRITICAL, Temperature, 104, °C), (CRITICAL, ..."


In [10]:
sensor_alert_count = sum(sum(level in ('WARNING', 'CRITICAL') for level, *_ in row) for row in equipment_window['condition'])
critical_alert_count = sum(sum(level == 'CRITICAL' for level, *_ in row) for row in equipment_window['condition'])
maintenance_due_count = equipment_window['maintenance_due'].str.lower().eq('yes').sum()
print('Sensor alerts:', sensor_alert_count)
print('Maintenance due records:', int(maintenance_due_count))
print('Equipment alerts (sensor + maintenance):', int(sensor_alert_count + maintenance_due_count))
print('Critical equipment alerts:', int(critical_alert_count))


Sensor alerts: 12
Maintenance due records: 4
Equipment alerts (sensor + maintenance): 16
Critical equipment alerts: 5
